# Speech-to-Text Model Comparison

## Install Required Packages

In [45]:
!pip install transformers librosa soundfile

## Import Libraries

In [46]:
import librosa
import torch
import IPython.display as display
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import numpy as np

## Load Model: OpenAI Whisper Base

In [47]:
from transformers import pipeline

whisper_base_pipeline = pipeline("automatic-speech-recognition", model="openai/whisper-base")

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

## Load Model: OpenAI Whisper Tiny

In [48]:
whisper_tiny_pipeline = pipeline("automatic-speech-recognition", model="openai/whisper-tiny")

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

## Load Model: Facebook Wav2Vec2 Large

In [49]:
wav2vec2_pipeline = pipeline("automatic-speech-recognition", model="facebook/wav2vec2-large-960h-lv60-self")

Loading weights:   0%|          | 0/423 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-large-960h-lv60-self
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Load Model: Facebook HuBERT Large

In [50]:
hubert_pipeline = pipeline("automatic-speech-recognition", model="facebook/hubert-large-ls960-ft")

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

## Load Model: Distil-Whisper Medium

In [51]:
distil_whisper_pipeline = pipeline("automatic-speech-recognition", model="distil-whisper/distil-medium.en")

Loading weights:   0%|          | 0/419 [00:00<?, ?it/s]

## Baseline Whisper Transcriber

In [52]:
from transformers import pipeline

transcriber = pipeline("automatic-speech-recognition", model="openai/whisper-base")

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

## Load and Play the Audio File

In [53]:
from google.colab import files

uploaded = files.upload()

In [54]:
import os
import glob

matching_files = glob.glob("*John*Wick*") + glob.glob("*.aac") + glob.glob("*.m4a")

if matching_files:
    audio_path = matching_files[0]
    audio, sampling_rate = librosa.load(audio_path, sr=16000)
    display.display(display.Audio(audio_path, autoplay=True))
else:
    sampling_rate = 16000
    t = np.linspace(0, 3, sampling_rate * 3, endpoint=False)
    audio = np.sin(2 * np.pi * 440 * t) * 0.1
    display.display(display.Audio(audio, rate=sampling_rate, autoplay=False))

/tmp/ipykernel_795/3724262756.py:8: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sampling_rate = librosa.load(audio_path, sr=16000)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


## Perform Baseline Speech-to-Text Transcription

In [55]:
result = transcriber(audio_path)

print(result["text"])

 Hi. Hello. One, two, three. I'm Zon Wake. Pew pew.


## Compare Transcriptions Across All Models

In [56]:
import pandas as pd

results = []
models_dict = {
    "OpenAI Whisper Base": whisper_base_pipeline,
    "OpenAI Whisper Tiny": whisper_tiny_pipeline,
    "Facebook Wav2Vec2 Large": wav2vec2_pipeline,
    "Facebook HuBERT Large": hubert_pipeline,
    "Distil-Whisper Medium": distil_whisper_pipeline
}

for name, model_pipeline in models_dict.items():
    try:
        out = model_pipeline(audio_path)
        text = out["text"]
    except Exception as e:
        text = f"Error: {str(e)}"
    results.append({"Model": name, "Transcription": text})

comparison_df = pd.DataFrame(results)
comparison_df

,Model,Transcription
0,OpenAI Whisper Base,"Hi. Hello. One, two, three. I'm Zon Wake. Pew..."
1,OpenAI Whisper Tiny,"Hi. Hello. One, two, three. I'm John Wake. Pe..."
2,Facebook Wav2Vec2 Large,HI HELLO ONE TWO THREE AMSON WAK PP
3,Facebook HuBERT Large,HIH HELLO ONE TWO THREE AMZON WAK PE PEW
4,Distil-Whisper Medium,"Hi, hello. One, two, three. I'm Zon Wick. Pew..."
